In [ ]:
import math
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Hyperparameters
NUM_LAYERS = 6
HIDDEN_DIM = 384
NUM_HEADS  = 6
COND_DIM   = 1  # log10(E_inc) from LemursDataset

# Calorimeter grid parameters (CaloChallenge Dataset 2 / LEMURS)
Z_LAYERS  = 45
R_BINS    = 9
PHI_BINS  = 16
VOXEL_DIM = R_BINS * PHI_BINS       # 144 features per layer
TOTAL_VOXELS = Z_LAYERS * VOXEL_DIM  # 6480

BATCH_SIZE    = 16
LEARNING_RATE = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset config — voxel_shape matches LemursDataset convention (D, H, W) = (Z, PHI, R)
CFG = dict(
    data_path    = "LEMURS_Par04SiW_gamma_100kEvents_1GeV1TeV_GPSflat_part1.h5",
    showers_key  = "showers",
    energies_key = "incident_energy",
    voxel_shape  = (Z_LAYERS, PHI_BINS, R_BINS),  # (45, 16, 9)
    n_samples    = None,
    batch_size   = BATCH_SIZE,
)

print(f"Using device: {DEVICE}")

In [ ]:
class LemursDataset(Dataset):
    """
    Loads 3-D calorimeter showers from an HDF5 file produced by LEMURS/CaloChallenge.

    Expected HDF5 layout:
        showers           float32  (N, D*H*W)  or  (N, H, W, D)
        incident_energies float32  (N, 1)

    Returns:
        x   : (1, D, H, W)  energy tensor, log-normalised to [-1, 1]
        cond: scalar         log10(E_inc / GeV)
    """

    def __init__(self, path: str, showers_key: str, energies_key: str,
                 voxel_shape: tuple, log_eps: float = 1e-6,
                 data_slice=None): # Added data_slice parameter
        super().__init__()
        self.voxel_shape = voxel_shape
        self.log_eps = log_eps

        with h5py.File(path, "r") as f:
            # Load the slice of data
            # If data_slice is None, load all data. Otherwise, apply the slice.
            if data_slice is None:
                raw_showers = f[showers_key][:]
                raw_energies = f[energies_key][:]
            else:
                raw_showers = f[showers_key][data_slice]
                raw_energies = f[energies_key][data_slice]

            self.energies = raw_energies.astype(np.float32)

        raw = raw_showers.astype(np.float32)
        raw = raw.reshape(-1, *voxel_shape)    # (N, D, H, W)

        # log-normalise: map sparse positive energy deposits to ~[-1, 1]
        raw = np.log(raw + log_eps)
        self.vmin, self.vmax = raw.min(), raw.max()
        self.showers = (raw - self.vmin) / (self.vmax - self.vmin) * 2 - 1  # [-1, 1]

    def __len__(self):
        return len(self.showers)

    def __getitem__(self, idx):
        x    = torch.from_numpy(self.showers[idx]).unsqueeze(0)  # (1, D, H, W)
        cond = math.log10(float(self.energies[idx]) + 1e-9)
        cond = torch.tensor(cond, dtype=torch.float32)
        return x, cond


def make_train_test_dataloaders(cfg: dict, train_split: float = 0.8):
    """
    Creates train and test DataLoaders with a static split.
    """
    # First, get the total number of samples from the HDF5 file
    with h5py.File(cfg["data_path"], "r") as f:
        total_samples = len(f[cfg["showers_key"]]) if cfg['n_samples'] is None else cfg['n_samples']


    # Calculate split index
    train_size = int(train_split * total_samples)
    # test_size = total_samples - train_size # Not strictly needed

    # Define slices for train and test
    train_slice = slice(0, train_size)
    test_slice = slice(train_size, total_samples)

    # Create datasets
    train_ds = LemursDataset(
        path        = cfg["data_path"],
        showers_key = cfg["showers_key"],
        energies_key= cfg["energies_key"],
        voxel_shape = cfg["voxel_shape"],
        data_slice  = train_slice,
    )
    test_ds = LemursDataset(
        path        = cfg["data_path"],
        showers_key = cfg["showers_key"],
        energies_key= cfg["energies_key"],
        voxel_shape = cfg["voxel_shape"],
        data_slice  = test_slice,
    )

    print(f"Train Dataset: {len(train_ds)} events | Test Dataset: {len(test_ds)} events | Voxel shape: {cfg['voxel_shape']}")

    # Create DataLoaders
    train_loader = DataLoader(
        train_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = True,
        num_workers = 2,
        pin_memory  = True,
        drop_last   = True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = False, # No need to shuffle test data
        num_workers = 8,
        pin_memory  = True,
        drop_last   = False, # Keep all test samples
    )

    return train_loader, test_loader

Dataset loaded with 128 synthetic showers.


In [ ]:
train_loader, test_loader = make_train_test_dataloaders(CFG, train_split=0.8)

In [3]:
class EDMDenoiser:
    """Implements Karras et al. (EDM) formulation components."""
    def __init__(self, sigma_data=0.5):
        self.sigma_data = sigma_data

    def get_scalings(self, sigma):
        """Calculates EDM framework scaling coefficients."""
        # Ensure sigma matches target dimensions for broadcasting
        sigma = sigma.view(-1, 1, 1)
        
        c_skip = self.sigma_data**2 / (sigma**2 + self.sigma_data**2)
        c_out = sigma * self.sigma_data / (sigma**2 + self.sigma_data**2).sqrt()
        c_in = 1.0 / (sigma**2 + self.sigma_data**2).sqrt()
        c_noise = 0.25 * torch.log(sigma)
        
        return c_skip, c_out, c_in, c_noise

    def loss_weight(self, sigma):
        """EDM loss weight lambda(sigma)."""
        return (sigma**2 + self.sigma_data**2) / (sigma * self.sigma_data)**2

In [4]:
class TimestepEmbedding(nn.Module):
    """Sinusoidal embeddings for EDM noise levels."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, sigma):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=sigma.device) * -emb)
        emb = sigma.view(-1, 1) * emb.view(1, -1)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb

class DiTBlock(nn.Module):
    """Transformer block utilizing Adaptive Layer Normalization (AdaLN) for conditioning."""
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        
        # AdaLN parameter projection (scale, shift, gate for both attention and MLP)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, 6 * hidden_dim)
        )

    def forward(self, x, cond_emb):
        # Unpack modulation parameters
        mod = self.adaLN_modulation(cond_emb).chunk(6, dim=-1)
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = mod
        
        # Expand chunks over sequence dimension
        shift_msa, scale_msa, gate_msa = shift_msa.unsqueeze(1), scale_msa.unsqueeze(1), gate_msa.unsqueeze(1)
        shift_mlp, scale_mlp, gate_mlp = shift_mlp.unsqueeze(1), scale_mlp.unsqueeze(1), gate_mlp.unsqueeze(1)

        # Attention block
        norm_x = self.norm1(x) * (1 + scale_msa) + shift_msa
        attn_out, _ = self.attn(norm_x, norm_x, norm_x)
        x = x + gate_msa * attn_out
        
        # MLP block
        norm_x = self.norm2(x) * (1 + scale_mlp) + shift_mlp
        x = x + gate_mlp * self.mlp(norm_x)
        return x

In [5]:
class CaloDiT(nn.Module):
    """Diffusion Transformer adapting longitudinal calorimeter layers as tokens."""
    def __init__(self, num_layers=6, hidden_dim=384, num_heads=6):
        super().__init__()
        # Treating each Z-layer (radial/phi slice) as a sequence token
        self.input_proj = nn.Linear(VOXEL_DIM, hidden_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, Z_LAYERS, hidden_dim))
        
        self.time_embed = TimestepEmbedding(hidden_dim // 2)
        self.cond_proj = nn.Sequential(
            nn.Linear((hidden_dim // 2) + COND_DIM, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.blocks = nn.ModuleList([DiTBlock(hidden_dim, num_heads) for _ in range(num_layers)])
        
        self.final_norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, 2 * hidden_dim))
        self.final_proj = nn.Linear(hidden_dim, VOXEL_DIM)
        
        nn.init.normal_(self.pos_embed, std=0.02)

    def forward(self, x, sigma, cond_vec):
        # Flatten radial & angular dimensions into token sequence
        B, Z, R, P = x.shape
        x = x.view(B, Z, R * P)
        x = self.input_proj(x) + self.pos_embed
        
        # Build global conditioning vector
        edm_denoiser = EDMDenoiser()
        _, _, _, c_noise = edm_denoiser.get_scalings(sigma)
        t_emb = self.time_embed(c_noise.squeeze(-1).squeeze(-1))
        
        global_cond = torch.cat([t_emb, cond_vec], dim=-1)
        cond_emb = self.cond_proj(global_cond)
        
        # Pass through DiT layers
        for block in self.blocks:
            x = block(x, cond_emb)
            
        # Output generation
        scale, shift = self.final_adaLN(cond_emb).chunk(2, dim=-1)
        x = self.final_norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        x = self.final_proj(x)
        
        return x.view(B, Z, R, P)

In [ ]:
# Initialization
model     = CaloDiT(num_layers=NUM_LAYERS, hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
denoiser  = EDMDenoiser()

model.train()
print("Starting training loop...")

for epoch in range(1):
    for step, (showers, conds) in enumerate(train_loader):
        showers, conds = showers.to(DEVICE), conds.to(DEVICE)

        # Dataset returns (B, 1, Z, PHI, R); CaloDiT expects (B, Z, R, PHI)
        showers = showers.squeeze(1).permute(0, 1, 3, 2).contiguous()  # (B, 45, 9, 16)
        conds   = conds.unsqueeze(-1)                                   # (B, 1)

        optimizer.zero_grad()

        # Sample EDM noise levels (log-normal, Karras et al.)
        log_sigma = torch.randn(showers.shape[0], device=DEVICE) * 1.2 - 1.2
        sigma     = torch.exp(log_sigma)

        c_skip, c_out, c_in, c_noise = denoiser.get_scalings(sigma)
        noise          = torch.randn_like(showers)
        noised_showers = showers + noise * sigma.view(-1, 1, 1, 1)

        model_input    = noised_showers * c_in.view(-1, 1, 1, 1)
        nn_out         = model(model_input, sigma, conds)

        denoised_output = c_skip.view(-1, 1, 1, 1) * noised_showers + c_out.view(-1, 1, 1, 1) * nn_out

        loss_weight = denoiser.loss_weight(sigma).view(-1, 1, 1, 1)
        loss        = (loss_weight * (denoised_output - showers) ** 2).mean()

        loss.backward()
        optimizer.step()

        if step % 2 == 0:
            print(f"Step {step:02d} | Loss: {loss.item():.4f}")

In [ ]:
@torch.no_grad()
def sample_edm(model, conds, steps=20, sigma_max=80.0, sigma_min=0.002, rho=7.0):
    """Generates synthetic calorimeter showers via deterministic Heun sampling."""
    model.eval()
    B = conds.shape[0]
    denoiser = EDMDenoiser()
    
    # Calculate time steps using the EDM power schedule coordinate space
    step_indices = torch.arange(steps, dtype=torch.float32, device=DEVICE)
    t_steps = (sigma_max ** (1 / rho) + step_indices / (steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([t_steps, torch.zeros_like(t_steps[:1])]) # Append 0 to the end
    
    # Initialize from pure noise
    x = torch.randn(B, Z_LAYERS, R_BINS, PHI_BINS, device=DEVICE) * t_steps[0]
    
    for i in range(steps):
        t_cur, t_next = t_steps[i], t_steps[i+1]
        
        # --- Form Euler step evaluation ---
        c_skip, c_out, c_in, _ = denoiser.get_scalings(t_cur)
        model_input = x * c_in
        nn_out = model(model_input, t_cur, conds)
        d_cur = (x - (c_skip * x + c_out * nn_out)) / t_cur
        
        x_next = x + (t_next - t_cur) * d_cur
        
        # --- Apply Heun's second-order correction (if not the last step) ---
        if t_next > 0:
            c_skip, c_out, c_in, _ = denoiser.get_scalings(t_next)
            model_input = x_next * c_in
            nn_out_next = model(model_input, t_next, conds)
            d_next = (x_next - (c_skip * x_next + c_out * nn_out_next)) / t_next
            
            x = x + (t_next - t_cur) * 0.5 * (d_cur + d_next)
        else:
            x = x_next
            
    return x

# Test inference execution
test_cond = torch.randn(2, COND_DIM, device=DEVICE)
generated_showers = sample_edm(model, test_cond, steps=10)
print(f"Successfully generated showers output structural shape: {generated_showers.shape}")